[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sysbiochalmers/mesbcourse/blob/main/exercises/GEM0_precourse_introduction/gem01.ipynb)

# GEM0: GEMs in cobrapy

A Python / cobrapy port of the RAVEN MATLAB exercise.

**Learning goals**

- Run commands in a Python notebook.
- Know what information is contained in a GEM.
- Interpret gene-protein-reaction (GPR) rules.
- Understand the meaning of a reaction's lower and upper bounds.
- Run flux balance analysis (FBA) and interpret the results.
- See with flux variability analysis (FVA) that many alternative optimal flux distributions exist.
- See that random sampling also provides knowledge on flux uncertainty.

**Setup.** Required packages: `cobra`, `pandas`, `numpy`, `matplotlib`, `openpyxl`. Place `yeast-GEM.xml` in the working directory. It can be downloaded from https://github.com/SysBioChalmers/yeast-GEM (look in the `model/` folder).

In [ ]:
import cobra
from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from cobra.sampling import sample
from cobra.util import create_stoichiometric_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cobra.__version__

## 1. Loading and inspecting a model

Genome-scale metabolic models come in many file formats, but the widely accepted one is SBML (Systems Biology Markup Language), usually with the `.xml` extension. We use the latest version, level 3, with the Flux Balance Constraint (FBC) package. SBML is not meant to be human-readable.

Load the model.

In [ ]:
model = read_sbml_model('yeast-GEM.xml')

Inspecting the model gives a summary of its size.

In [ ]:
model

The model contains lists of reactions, metabolites, and genes.

In [ ]:
len(model.reactions)

In [ ]:
len(model.metabolites)

In [ ]:
len(model.genes)

It also has a small number of compartments.

In [ ]:
model.compartments

### Looking at a single reaction

Reactions, metabolites, and genes can be looked up by id with `get_by_id`. cobrapy also allows dot notation when the id is a valid Python identifier.

In [ ]:
model.reactions.get_by_id('r_2157')

Equivalent, using dot notation:

In [ ]:
model.reactions.r_2157

The reaction object has a name, an equation, a GPR rule, a subsystem, and a set of compartments. Each is accessed as an attribute.

In [ ]:
model.reactions.r_2157.name

In [ ]:
model.reactions.r_2157.reaction

In [ ]:
model.reactions.r_2157.compartments

In [ ]:
model.reactions.r_2157.subsystem

Iterating over `reaction.metabolites` gives the stoichiometric coefficients.

In [ ]:
for met, coef in model.reactions.r_2157.metabolites.items():
    print(f'{coef:+g}  {met.id}  ({met.name}) [{met.compartment}]')

**Question 1.** In which organelle is `r_2157` localised?

### Gene-protein-reaction (GPR) associations

The GPR rule describes which genes encode the protein(s) catalysing a reaction. `and` between two gene ids means both are required (an enzyme complex). `or` means either is sufficient (isozymes).

In [ ]:
model.reactions.r_2157.gene_reaction_rule

In [ ]:
model.reactions.r_2142.gene_reaction_rule

In [ ]:
model.reactions.r_0001.gene_reaction_rule

**Question 2.** Can you explain the gene association of `r_2157`? Compare with `r_2142` (a different structural relationship) and `r_0001` (more complex).

Genes also know which reactions they catalyse, via `gene.reactions`.

In [ ]:
for r in model.genes.YCR034W.reactions:
    print(f'{r.id}  {r.name}')

In [ ]:
for r in model.genes.YLR372W.reactions:
    print(f'{r.id}  {r.name}')

**Question 3.** One of the genes associated to `r_2157` also seems to catalyse a very different reaction. Which is it? Can you find literature evidence that the protein coded by this gene is involved in that additional catalytic activity?

### Looking at the raw SBML

**Question 4.** Open `yeast-GEM.xml` in a text editor and search for `r_2157`. Can you easily fetch the same information that the reaction object gave you above?

### Optional: export to a spreadsheet

cobrapy has no built-in Excel export, but pandas does. Build a DataFrame from the reaction attributes and write it to disk.

In [ ]:
rxns_df = pd.DataFrame([{
    'ID': r.id,
    'name': r.name,
    'equation': r.reaction,
    'GPR': r.gene_reaction_rule,
    'lb': r.lower_bound,
    'ub': r.upper_bound,
    'subsystem': r.subsystem,
} for r in model.reactions])
rxns_df.head()

In [ ]:
rxns_df.to_excel('yeast-rxns.xlsx', index=False)

## 2. Model structure

In MATLAB / RAVEN, the model is a struct with parallel arrays (`model.rxns`, `model.lb`, `model.ub`, `model.S`, ...). In cobrapy the model is an object: `model.reactions`, `model.metabolites`, `model.genes` are lists of objects, and the stoichiometric matrix is built on demand. Bounds, GPRs, and objective coefficients live on each reaction object.

Lists support slicing.

In [ ]:
model.reactions[:5]

In [ ]:
model.metabolites[:5]

In [ ]:
model.genes[:5]

The stoichiometric matrix `S` is built with one function call. Rows are metabolites, columns are reactions.

In [ ]:
S = create_stoichiometric_matrix(model)
S.shape

### Bounds and objective

Each reaction carries its own lower bound, upper bound, and objective coefficient. We look at the glucose exchange reaction `r_1714`.

In [ ]:
model.reactions.r_1714

In [ ]:
model.reactions.r_1714.lower_bound

In [ ]:
model.reactions.r_1714.upper_bound

In [ ]:
model.reactions.r_1714.objective_coefficient

**Question 5.** What do `lower_bound`, `upper_bound`, and `objective_coefficient` indicate? What values can they take, and what does this mean?

The model's current objective:

In [ ]:
model.objective

### Removing a reaction

cobrapy can wrap changes to the model in a `with model:` context. Changes inside the block are rolled back automatically on exit. This is more idiomatic (and faster) than copying the model for temporary changes.

In [ ]:
len(model.reactions)

In [ ]:
with model:
    model.remove_reactions(['r_1237'])
    print('Inside the with-block:', len(model.reactions), 'reactions')

print('Outside the with-block: ', len(model.reactions), 'reactions')

**Question 6.** When you delete one reaction from the model, what other parts of the model change? (Assume the removed reaction does not introduce unique metabolites, genes, or compartments.)

## 3. Flux balance analysis

We now assume that the yeast aims to grow as fast as possible: set the biomass reaction `r_4041` as the objective.

In [ ]:
model.objective = 'r_4041'

In [ ]:
model.reactions.r_4041

**Question 7.** What are the substrates and products of `r_4041`?

This model has a nested biomass equation: separate macromolecule reactions feed into `r_4041`.

In [ ]:
for rid in ['r_4041', 'r_2108', 'r_4047', 'r_4048', 'r_4049',
            'r_4050', 'r_4063', 'r_4065', 'r_4598', 'r_4599']:
    r = model.reactions.get_by_id(rid)
    print(f'{rid}  ({r.name})')
    print(f'    {r.reaction}')

**Question 8.** In reaction `r_4041`, what does the use of ATP as substrate represent? And what determines the stoichiometric coefficient?

### Constraining glucose uptake

Limit glucose uptake to 0.5 mmol/gDCW/h.

In [ ]:
model.reactions.r_1714

In [ ]:
model.reactions.r_1714.lower_bound = -0.5

In [ ]:
model.reactions.r_1714

**Question 9.** Pay attention: to limit glucose *uptake*, we set the *lower* bound to `-0.5`. Why not the upper bound to `0.5`? Hint: look at the reaction equation.

### Run FBA

In [ ]:
solution = model.optimize()

In [ ]:
solution

In [ ]:
solution.objective_value

`model.summary()` aggregates the result: growth rate, uptake fluxes, and secretion fluxes.

In [ ]:
model.summary()

**Question 10.** What is the meaning of the objective value here? Consider what objective function we set.

**Question 11.** What growth rate can be reached? What are the other exchange rates?

All reaction fluxes are available as a pandas Series:

In [ ]:
solution.fluxes.head()

Including the reaction we previously removed inside the `with` block, which is back in the model:

In [ ]:
solution.fluxes['r_1237']

## 4. Flux variability analysis

FBA returns one feasible flux distribution at the optimum, but many alternatives exist. FVA gives, for each reaction, the minimum and maximum flux compatible with the optimum (or with a given fraction of it).

First, fix glucose uptake and growth at their measured values, allowing 5% variance. cobrapy has no direct equivalent of RAVEN's `setParam(..., 'var', ..., 5)`, so we write a small helper.

In [ ]:
def set_var(model, rxn_id, value, percent=5):
    delta = abs(value) * percent / 100.0
    model.reactions.get_by_id(rxn_id).bounds = (value - delta, value + delta)

In [ ]:
model_fva = model.copy()
model_fva.objective = 'r_4041'
set_var(model_fva, 'r_1714', -0.5)
sol0 = model_fva.optimize()
sol0.objective_value

In [ ]:
set_var(model_fva, 'r_4041', sol0.objective_value)

Now run FVA. `fraction_of_optimum=0.95` allows growth to drop to 95% of the optimum.

In [ ]:
fva = flux_variability_analysis(model_fva, fraction_of_optimum=0.95)

In [ ]:
fva.head()

The variability range of each reaction is `maximum - minimum`.

In [ ]:
var_range = (fva['maximum'] - fva['minimum']).sort_values()

In [ ]:
(var_range < 1e-10).mean()

In [ ]:
(var_range > 1999).mean()

Most reactions are tightly constrained, but a small fraction can take any flux in the solver's default range. Plotting the cumulative distribution makes this concrete.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(var_range.values, np.arange(len(var_range)) / len(var_range))
plt.xscale('symlog', linthresh=1e-6)
plt.xlabel('Flux variability range  (max - min)')
plt.ylabel('Cumulative fraction of reactions')
plt.title('CDF of flux variability')
plt.grid(True, alpha=0.3)
plt.show()

**Question 12.** Can you think of two reasons why a reaction would have a variability of 0?

Which reactions have very large variability?

In [ ]:
highest_var = var_range[var_range > 1999].index.tolist()
for rid in highest_var[:5]:
    print(f'{rid}\t{model_fva.reactions.get_by_id(rid).name}')

## 5. Random sampling

FVA returns extreme values; many are biologically unrealistic. Random sampling draws flux distributions from the feasible polytope, giving means and standard deviations that are often more informative.

Suppose we cultivated yeast in a bioreactor and measured:

- glucose uptake: 0.5 mmol/gDCW/h  (`r_1714` = -0.5)
- CO2 production: 1.18 mmol/gDCW/h (`r_1672` = 1.18)
- O2 consumption: 1.27 mmol/gDCW/h (`r_1992` = -1.27)
- growth rate:    0.0394 /h         (`r_4041` = 0.0394)

Allow 5% variance on each measurement.

In [ ]:
model_rs = model.copy()
model_rs.objective = 'r_4041'
set_var(model_rs, 'r_1714', -0.5)
set_var(model_rs, 'r_1672',  1.18)
set_var(model_rs, 'r_1992', -1.27)
set_var(model_rs, 'r_4041',  0.0394)

In [ ]:
for rid in ['r_1714', 'r_1672', 'r_1992', 'r_4041']:
    r = model_rs.reactions.get_by_id(rid)
    print(f'{rid}: bounds = ({r.lower_bound:.4f}, {r.upper_bound:.4f})')

cobrapy's default sampler is OptGP. ACHR is also available. Numerical values differ between samplers and between runs; qualitative conclusions are the same.

In [ ]:
samples = sample(model_rs, n=1000, method='optgp')

In [ ]:
samples.shape

In [ ]:
samples.iloc[:5, :5]

Compute mean and standard deviation of each reaction's flux. Setting near-zero fluxes to exactly zero gives a cleaner summary.

In [ ]:
samples_clean = samples.where(samples.abs() >= 1e-10, 0.0)
summary = pd.DataFrame({
    'mean': samples_clean.mean(),
    'sd':   samples_clean.std(),
    'min':  samples_clean.min(),
    'max':  samples_clean.max(),
})
summary.head()

How do the reactions that had high FVA variability behave under sampling?

In [ ]:
summary.loc[highest_var[:10]]

**Question 13.** Looking at the fluxes, is there anything striking? For the reactions with FVA variability around 2000 (Question 12), how big are their mean and standard deviation under sampling?

## Notes on differences from the RAVEN version

- **SBML inspection (Q4)** is identical: open `yeast-GEM.xml` in a text editor.
- **Model structure (Q5, Q6)**: cobrapy uses objects, not parallel arrays. The information content is the same, just accessed differently. `model.remove_reactions(['r_1237'])` updates everything attached to that reaction in one call.
- **Random sampling (Q13)**: cobrapy uses OptGP (or ACHR) rather than RAVEN's sampler, so exact numerical values differ between toolboxes and between runs. The qualitative observations (many reactions carry zero flux, sd often exceeds the mean, FVA extremes are not representative of typical flux values) still hold.
- **Solvers**: cobrapy uses optlang and picks GLPK by default if Gurobi or CPLEX are not installed. To switch: `model.solver = 'gurobi'`.